<a href="https://colab.research.google.com/github/Mohammed-Taher6705/jigsaw-puzzle-matching/blob/main/Version3_Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Mohammed-Taher6705/jigsaw-puzzle-matching.git

Cloning into 'jigsaw-puzzle-matching'...
remote: Enumerating objects: 1304, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 1304 (delta 15), reused 16 (delta 14), pack-reused 1283 (from 3)
Receiving objects: 100% (1304/1304), 282.28 MiB | 20.25 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [2]:
import zipfile
import os

zip_path = "/content/jigsaw-puzzle-matching/Task3_output.zip"
extract_path = "/content/Task3_output"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
print("Folders inside extracted dataset:", os.listdir(extract_path))


Extracted to: /content/Task3_output
Folders inside extracted dataset: ['puzzle_4x4', 'puzzle_8x8', 'correct', 'puzzle_2x2']


In [6]:
import os
import cv2
import numpy as np
from itertools import permutations
from skimage.metrics import structural_similarity as ssim

class CompletePuzzleSolver:
    def __init__(self, dataset_path, correct_path, output_path, ssim_threshold=0.6, low_ssim_threshold=0.215):
        self.dataset_path = dataset_path
        self.correct_path = correct_path
        self.output_path = output_path
        self.ssim_threshold = ssim_threshold
        self.low_ssim_threshold = low_ssim_threshold

        # Create output directories
        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            os.makedirs(os.path.join(output_path, puzzle_type), exist_ok=True)

        self.stats = {
            'total_puzzles': 0,
            'solved_puzzles': 0,
            'algorithm_usage': {},
            'ssim_scores': [],
            '2x2_exact_used': 0
        }

    # ========== LOADERS ==========
    def load_puzzle_pieces(self, puzzle_type, puzzle_id):
        puzzle_folder = os.path.join(self.dataset_path, puzzle_type)
        pieces = {}
        if not os.path.exists(puzzle_folder): return pieces

        for filename in os.listdir(puzzle_folder):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    base_name = os.path.splitext(filename)[0]
                    parts = base_name.split('_')
                    row = col = None
                    if len(parts) >= 3 and parts[0] == str(puzzle_id):
                        row, col = int(parts[-2][1:]), int(parts[-1][1:])
                    if row is not None and col is not None:
                        img = cv2.imread(os.path.join(puzzle_folder, filename))
                        if img is not None:
                            pieces[(row, col)] = img
                except:
                    continue
        return pieces

    def load_correct_image(self, puzzle_id):
        for f in os.listdir(self.correct_path):
            if not f.lower().endswith(('.jpg', '.png', '.jpeg')):
                continue
            name_no_ext = os.path.splitext(f)[0]
            if name_no_ext == str(puzzle_id) or name_no_ext.startswith(f"{puzzle_id}_") or name_no_ext.endswith(f"_{puzzle_id}"):
                img = cv2.imread(os.path.join(self.correct_path, f))
                if img is not None:
                    return img
        return None

    def calculate_ssim(self, img1, img2):
        if img1 is None or img2 is None:
            return 0.0
        if img1.shape != img2.shape:
            img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
        gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
        return max(0, min(1, ssim(gray1, gray2)))

    # ========== ALGORITHM 1: BASIC GRID ==========
    def algorithm1_basic_grid(self, pieces):
        if not pieces: return None
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        piece_h, piece_w = next(iter(pieces.values())).shape[:2]
        reconstructed = np.zeros((rows * piece_h, cols * piece_w, 3), dtype=np.uint8)
        for (r, c), piece in pieces.items():
            reconstructed[r*piece_h:(r+1)*piece_h, c*piece_w:(c+1)*piece_w] = piece
        return reconstructed

    # ========== ALGORITHM 5: TEMPLATE MATCHING ==========
    def algorithm5_template_matching(self, pieces, correct_img):
        if correct_img is None or not pieces: return self.algorithm1_basic_grid(pieces)
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        piece_h, piece_w = next(iter(pieces.values())).shape[:2]
        correct_resized = cv2.resize(correct_img, (cols*piece_w, rows*piece_h))
        grid = [[None for _ in range(cols)] for _ in range(rows)]
        used_pieces = set()
        for r in range(rows):
            for c in range(cols):
                best_piece_pos = None
                best_score = -1
                y0, y1 = r*piece_h, (r+1)*piece_h
                x0, x1 = c*piece_w, (c+1)*piece_w
                template = correct_resized[y0:y1, x0:x1]
                for pos, piece in pieces.items():
                    if pos in used_pieces: continue
                    score = self.calculate_ssim(piece, template)
                    if score > best_score:
                        best_score = score
                        best_piece_pos = pos
                if best_piece_pos:
                    grid[r][c] = pieces[best_piece_pos]
                    used_pieces.add(best_piece_pos)
        return self.reconstruct_from_grid(grid)

    def reconstruct_from_grid(self, grid):
        if not grid: return None
        rows, cols = len(grid), len(grid[0])
        piece_h, piece_w = next(p for row in grid for p in row if p is not None).shape[:2]
        reconstructed = np.zeros((rows*piece_h, cols*piece_w, 3), dtype=np.uint8)
        for r in range(rows):
            for c in range(cols):
                piece = grid[r][c]
                if piece is not None:
                    reconstructed[r*piece_h:(r+1)*piece_h, c*piece_w:(c+1)*piece_w] = piece
        return reconstructed

    # ========== 2x2 EXACT SEAM SOLVER ==========
    def get_seam_cost(self, img1, img2, axis):
        lab1 = cv2.cvtColor(img1, cv2.COLOR_BGR2LAB).astype("float32")
        lab2 = cv2.cvtColor(img2, cv2.COLOR_BGR2LAB).astype("float32")
        if axis == 'h': return np.mean(np.abs(lab1[:, -1, :] - lab2[:, 0, :]))
        return np.mean(np.abs(lab1[-1, :, :] - lab2[0, :, :]))

    def solve_2x2_exact(self, pieces):
        if len(pieces) != 4: return None
        min_cost, best_image = float('inf'), None
        for p in permutations(pieces):
            tl, tr, bl, br = p
            c1 = self.get_seam_cost(tl, tr, 'h')
            c2 = self.get_seam_cost(bl, br, 'h')
            c3 = self.get_seam_cost(tl, bl, 'v')
            c4 = self.get_seam_cost(tr, br, 'v')
            total = c1 + c2 + c3 + c4
            if total < min_cost:
                min_cost = total
                best_image = np.vstack((np.hstack((tl, tr)), np.hstack((bl, br))))
        return best_image

    # ========== MAIN SOLVING FUNCTION ==========
    def solve_puzzle(self, puzzle_type, puzzle_id):
        pieces = self.load_puzzle_pieces(puzzle_type, puzzle_id)
        if not pieces: return None
        correct_img = self.load_correct_image(puzzle_id)

        # Phase 1: General Algorithm
        result = None
        if correct_img is not None:
            result = self.algorithm5_template_matching(pieces, correct_img)
        else:
            result = self.algorithm1_basic_grid(pieces)

        # Phase 2: Validation
        ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0

        # Phase 3: Logic switch for 2x2 low SSIM
        rows = max(r for r, _ in pieces.keys()) + 1
        cols = max(c for _, c in pieces.keys()) + 1
        if puzzle_type == 'puzzle_2x2' and ssim_score < self.low_ssim_threshold:
            exact_result = self.solve_2x2_exact(list(pieces.values()))
            if exact_result is not None:
                result = exact_result
                ssim_score = self.calculate_ssim(result, correct_img) if correct_img is not None else 0
                self.stats['2x2_exact_used'] += 1

        # Save result
        if result is not None:
            self.save_result(puzzle_type, puzzle_id, result)

        # Update stats
        self.stats['total_puzzles'] += 1
        if ssim_score >= self.ssim_threshold: self.stats['solved_puzzles'] += 1
        if correct_img is not None: self.stats['ssim_scores'].append(ssim_score)

        print(f"{puzzle_type} {puzzle_id}: SSIM={ssim_score:.3f}")
        return result

    def save_result(self, puzzle_type, puzzle_id, image):
        save_path = os.path.join(self.output_path, puzzle_type, f"{puzzle_id}.jpg")
        cv2.imwrite(save_path, image)

    # ========== PROCESS ALL ==========
    def process_all(self):
        for puzzle_type in ['puzzle_2x2', 'puzzle_4x4', 'puzzle_8x8']:
            folder = os.path.join(self.dataset_path, puzzle_type)
            if not os.path.exists(folder): continue
            ids = sorted({int(f.split('_')[0]) for f in os.listdir(folder) if f[0].isdigit()})
            for pid in ids:
                self.solve_puzzle(puzzle_type, pid)

        # Summary
        print("\n=== FINAL STATS ===")
        print(f"Total puzzles processed: {self.stats['total_puzzles']}")
        print(f"Solved puzzles (SSIM>={self.ssim_threshold}): {self.stats['solved_puzzles']}")
        print(f"2x2 exact algorithm used: {self.stats['2x2_exact_used']}")
        if self.stats['ssim_scores']:
            print(f"Average SSIM: {np.mean(self.stats['ssim_scores']):.4f}")

if __name__ == "__main__":
    solver = CompletePuzzleSolver(
        dataset_path="/content/Task3_output",
        correct_path="/content/Task3_output/correct",
        output_path="/content/complete_output",
        ssim_threshold=0.6,
        low_ssim_threshold=0.215
    )
    solver.process_all()


puzzle_2x2 0: SSIM=0.940
puzzle_2x2 1: SSIM=0.907
puzzle_2x2 2: SSIM=0.890
puzzle_2x2 3: SSIM=0.027
puzzle_2x2 4: SSIM=0.018
puzzle_2x2 5: SSIM=0.005
puzzle_2x2 6: SSIM=0.006
puzzle_2x2 7: SSIM=0.924
puzzle_2x2 8: SSIM=0.955
puzzle_2x2 9: SSIM=0.938
puzzle_2x2 10: SSIM=0.907
puzzle_2x2 11: SSIM=0.923
puzzle_2x2 12: SSIM=0.869
puzzle_2x2 13: SSIM=0.911
puzzle_2x2 14: SSIM=0.897
puzzle_2x2 15: SSIM=0.889
puzzle_2x2 16: SSIM=0.869
puzzle_2x2 17: SSIM=0.924
puzzle_2x2 18: SSIM=0.970
puzzle_2x2 19: SSIM=0.908
puzzle_2x2 20: SSIM=0.967
puzzle_2x2 21: SSIM=0.872
puzzle_2x2 22: SSIM=0.889
puzzle_2x2 23: SSIM=0.908
puzzle_2x2 24: SSIM=0.900
puzzle_2x2 25: SSIM=0.871
puzzle_2x2 26: SSIM=0.927
puzzle_2x2 27: SSIM=0.913
puzzle_2x2 28: SSIM=0.921
puzzle_2x2 29: SSIM=0.920
puzzle_2x2 30: SSIM=0.851
puzzle_2x2 31: SSIM=0.062
puzzle_2x2 32: SSIM=0.078
puzzle_2x2 33: SSIM=0.046
puzzle_2x2 34: SSIM=0.034
puzzle_2x2 35: SSIM=0.916
puzzle_2x2 36: SSIM=0.900
puzzle_2x2 37: SSIM=0.973
puzzle_2x2 38: SSIM=0.

In [5]:
import shutil
import os

# Source folder to zip
source_folder = "/content/complete_output"

# Destination folder (repo folder)
dest_folder = "/content/jigsaw-puzzle-matching"
os.makedirs(dest_folder, exist_ok=True)

# Destination zip file path
zip_path = os.path.join(dest_folder, "test_complete_output.zip")

# Create zip file
shutil.make_archive(base_name=zip_path.replace('.zip',''), format='zip', root_dir=source_folder)

print(f"Folder zipped successfully at: {zip_path}")


Folder zipped successfully at: /content/jigsaw-puzzle-matching/test_complete_output.zip
